In [ ]:
"""
=============================================================
FILE 42 — GOAL DECOMPOSITION PATTERN
=============================================================

CONCEPTS TAUGHT
----------------
1. Goal Decomposition
2. Subgoal Generation
3. Hierarchical Planning
4. Autonomous Planning
5. Task Breakdown
6. AI Planning Systems
7. Multi-Step Objectives
8. Strategic Reasoning
9. Workflow Planning
10. Agentic Task Management

CORE IDEA
-----------
Break a large goal into
smaller manageable subgoals.

FLOW
-----
Large Goal
   ↓
Subgoal 1
Subgoal 2
Subgoal 3
   ↓
Execute Subgoals
   ↓
Final Strategy

REAL WORLD USE CASES
---------------------
- AI project planning
- Autonomous coding agents
- Enterprise planning
- Strategic execution systems
"""

# ============================================================
# STEP 1 — IMPORTS
# ============================================================

import os

from dotenv import load_dotenv

from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END

from IPython.display import Image, display

# ============================================================
# STEP 2 — ENV VARIABLES
# ============================================================

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# ============================================================
# STEP 3 — LLM
# ============================================================

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

# ============================================================
# STEP 4 — STATE
# ============================================================

class State(TypedDict):

    main_goal: str

    subgoal_1: str
    subgoal_2: str
    subgoal_3: str

    execution_plan: str

# ============================================================
# STEP 5 — SUBGOAL GENERATION
# ============================================================

def decompose_goal(state: State):

    response = llm.invoke(
        f"""
        Break the following business goal
        into 3 strategic subgoals.

        GOAL:
        {state['main_goal']}
        """
    )

    decomposition = response.content.split("\n")

    return {
        "subgoal_1": decomposition[0] if len(decomposition) > 0 else "",
        "subgoal_2": decomposition[1] if len(decomposition) > 1 else "",
        "subgoal_3": decomposition[2] if len(decomposition) > 2 else ""
    }

# ============================================================
# STEP 6 — EXECUTION NODE
# ============================================================

def execution_planner(state: State):

    response = llm.invoke(
        f"""
        Create an execution strategy for:

        SUBGOAL 1:
        {state['subgoal_1']}

        SUBGOAL 2:
        {state['subgoal_2']}

        SUBGOAL 3:
        {state['subgoal_3']}
        """
    )

    return {
        "execution_plan": response.content
    }

# ============================================================
# STEP 7 — BUILD GRAPH
# ============================================================

builder = StateGraph(State)

builder.add_node("decompose_goal", decompose_goal)

builder.add_node(
    "execution_planner",
    execution_planner
)

# ============================================================
# STEP 8 — DEFINE EDGES
# ============================================================

builder.add_edge(
    START,
    "decompose_goal"
)

builder.add_edge(
    "decompose_goal",
    "execution_planner"
)

builder.add_edge(
    "execution_planner",
    END
)

# ============================================================
# STEP 9 — COMPILE GRAPH
# ============================================================

graph = builder.compile()

display(
    Image(
        graph.get_graph().draw_mermaid_png()
    )
)

# ============================================================
# STEP 10 — RUN WORKFLOW
# ============================================================

result = graph.invoke(
    {
        "main_goal":
        """
        Build a successful AI-powered eCommerce platform
        """
    }
)

# ============================================================
# STEP 11 — PRINT RESULT
# ============================================================

print("\nEXECUTION PLAN\n")
print("=" * 60)

print(result["execution_plan"])